# Notebook 5 — Integration: Enzyme Active Sites and Structure–Function Relationships

## Welcome to the Capstone Notebook

In the first four notebooks we built up a complete picture of protein structure:

| Notebook | Level | What We Studied |
|----------|-------|-----------------|
| 1 | Primary | Amino acid sequence and composition |
| 2 | Secondary | α-helices, β-sheets, and coils |
| 3 | Tertiary | 3-D folding, domains, disulfide bonds |
| 4 | Quaternary | Subunit assembly and cooperative binding |

Now we put it all together by studying **enzyme active sites** — the precise three-dimensional arrangement of a small number of amino acids that gives an enzyme its catalytic power.

### Our Model System: Serine Proteases

A **protease** is an enzyme that cuts peptide bonds to degrade or process proteins.
A **serine protease** uses a **serine residue as the nucleophilic "chemical fist"** that attacks the peptide bond.

Serine proteases include:
- **Trypsin** — digestive enzyme; cuts after Arg and Lys
- **Chymotrypsin** — digestive enzyme; cuts after large hydrophobic residues (Phe, Trp, Tyr)
- **Thrombin** — blood-clotting cascade
- **Elastase** — lung and tissue remodeling

Although these enzymes have different substrate preferences, they all share the same catalytic mechanism — a remarkable example of **conserved molecular logic**.

### The Catalytic Triad

At the heart of every serine protease is a trio of residues called the **catalytic triad**:

| Residue | Role | Why It Matters |
|---------|------|----------------|
| **Ser 195** | Nucleophile — attacks the peptide bond | Hydroxyl (-OH) becomes reactive when deprotonated |
| **His 57** | Proton shuttle | Accepts H from Ser 195; donates H to leaving group |
| **Asp 102** | Electrostatic stabilizer | Negative charge orients and stabilizes His 57 |

These residue numbers follow **chymotrypsin numbering**, used by convention for the entire family.

### Learning Objectives

By the end of this notebook you will be able to:
1. Load and explore a protein structure programmatically (review)
2. Locate specific catalytic residues by number and measure distances between them
3. Compare sequences from two related enzymes and visualize conservation
4. Run a master analysis pipeline on an unknown enzyme
5. Explain the concept of **convergent evolution** using structural evidence

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Install and import everything we need
# Run this cell first, then use Runtime > Restart Runtime, then run all cells.
# ─────────────────────────────────────────────────────────────────────────────

# Install libraries — nglview is pinned to 3.0.8, the version verified to
# work in Colab. Later versions have rendering issues in this environment.
!pip install -q biopython nglview==3.0.8

# Enable the custom widget manager AFTER install so it registers the
# correct (pinned) version of the nglview widget.
from google.colab import output
output.enable_custom_widget_manager()

# ── Standard library imports ─────────────────────────────────────────────────
import os
import math
import warnings
warnings.filterwarnings('ignore')

# ── BioPython imports ────────────────────────────────────────────────────────
from Bio.PDB import PDBList, PDBParser, PPBuilder
from Bio import pairwise2

# ── Data and plotting imports ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 3-D visualization ────────────────────────────────────────────────────────
import nglview as nv

print("All libraries loaded successfully!")
print(f"nglview version: {nv.__version__}")

# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS — used throughout the notebook
# ─────────────────────────────────────────────────────────────────────────────

def calculate_distance(atom1, atom2):
    '''Calculate the straight-line (Euclidean) distance between two atoms.

    Parameters
    ----------
    atom1, atom2 : Bio.PDB.Atom objects

    Returns
    -------
    float : distance in Angstroms
    '''
    diff_vector = atom1.coord - atom2.coord
    distance = math.sqrt(sum(diff_vector ** 2))
    return round(distance, 2)


def classify_residue(residue_name):
    '''Classify an amino acid residue by its side-chain chemistry.

    Parameters
    ----------
    residue_name : str  — three-letter amino acid code (e.g. 'ALA', 'GLU')

    Returns
    -------
    str : one of 'Nonpolar', 'Polar', 'Positive', 'Negative'
    '''
    nonpolar  = {'ALA', 'VAL', 'LEU', 'ILE', 'PRO', 'PHE', 'TRP', 'MET', 'GLY'}
    polar     = {'SER', 'THR', 'CYS', 'TYR', 'ASN', 'GLN', 'HIS'}
    positive  = {'LYS', 'ARG'}
    negative  = {'ASP', 'GLU'}

    if residue_name in nonpolar:
        return 'Nonpolar'
    elif residue_name in polar:
        return 'Polar'
    elif residue_name in positive:
        return 'Positive'
    elif residue_name in negative:
        return 'Negative'
    else:
        return 'Other'

print("Helper functions defined: calculate_distance(), classify_residue()")

---
## Section 1 — Loading and Exploring Trypsin (PDB: 1TGN)

### Why 1TGN?

**1TGN** is a structure of bovine **trypsin**, solved by X-ray crystallography.
It contains a single polypeptide chain with the complete catalytic machinery intact,
making it a clean starting point for active-site analysis.

### What Is "Resolution" in Crystallography?

When scientists solve a protein structure by X-ray crystallography, the **resolution** (measured in Ångströms, Å) tells us how much detail we can see:

| Resolution | What It Means |
|-----------|---------------|
| < 1.5 Å | Atomic resolution — individual atoms clearly visible |
| 1.5–2.5 Å | High resolution — side chains well-defined |
| 2.5–3.5 Å | Medium resolution — backbone clear, side chains approximate |
| > 3.5 Å | Low resolution — rough shape only |

1 Å = 0.1 nanometers = 0.0000000001 meters — about the radius of a hydrogen atom!

### Modular Code Design

In this notebook we use a **modular** approach: we define reusable functions rather than repeating code.
This is how real bioinformatics software is written.

> **Think of functions like a lab protocol** — written once, run on any sample you choose.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Define a reusable function to load any PDB structure
# ─────────────────────────────────────────────────────────────────────────────

def load_structure(pdb_id, file_dir='.'):
    '''Download a PDB structure and return BioPython objects for it.

    Parameters
    ----------
    pdb_id   : str  — 4-character PDB accession code (e.g. '1TGN')
    file_dir : str  — directory where the downloaded file will be saved

    Returns
    -------
    structure : Bio.PDB.Structure  — the full structure object
    model     : Bio.PDB.Model      — the first model (index 0); for X-ray
                                     structures there is usually only one model
    pdb_path  : str                — file path to the downloaded PDB file
    '''
    # Step 1: Create a PDBList object to handle downloading from the RCSB
    pdb_downloader = PDBList()

    # Step 2: Download the PDB file (if not already downloaded)
    #   obsolete=False  → don't look in the obsolete archive
    #   file_format='pdb' → download the legacy PDB format (not mmCIF)
    # The try/except block catches network errors and gives a clear message
    try:
        pdb_path = pdb_downloader.retrieve_pdb_file(
            pdb_id,
            obsolete=False,
            pdir=file_dir,
            file_format='pdb'
        )
    except Exception as download_error:
        raise RuntimeError(
            f'Could not download {pdb_id} from RCSB PDB.\n'
            'Check your internet connection and try again.\n'
            f'Original error: {download_error}'
        )

    # Step 3: Create a PDBParser and load the downloaded file
    #   QUIET=True suppresses minor warnings about non-standard residues
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)

    # Step 4: Extract the first (and usually only) model
    model = structure[0]

    print(f"Successfully loaded {pdb_id}")
    print(f"  File: {pdb_path}")
    print(f"  Models: {len(list(structure.get_models()))}")
    print(f"  Chains: {[chain.id for chain in model.get_chains()]}")

    return structure, model, pdb_path


print("Function load_structure() defined and ready to use.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Load trypsin (1TGN) and explore its basic properties
# ─────────────────────────────────────────────────────────────────────────────

# Load the structure using our new function
trypsin_structure, trypsin_model, trypsin_path = load_structure('1TGN')

print()
print("=" * 55)
print("TRYPSIN STRUCTURE SUMMARY (PDB: 1TGN)")
print("=" * 55)

# ── Count chains ──────────────────────────────────────────────────────────────
chain_list = list(trypsin_model.get_chains())
print(f"Number of chains: {len(chain_list)}")

for chain in chain_list:
    # Count only standard amino acid residues (ignore water, ligands)
    amino_acids = [res for res in chain.get_residues()
                   if res.get_id()[0] == ' ']   # ' ' = standard residue flag
    print(f"  Chain {chain.id}: {len(amino_acids)} amino acid residues")

# ── Count total atoms ─────────────────────────────────────────────────────────
all_atoms = list(trypsin_model.get_atoms())
print(f"Total atoms in model: {len(all_atoms)}")

# ── Identify the main trypsin polypeptide chain ───────────────────────────────
# Chain labels can differ between PDB file versions, so we detect the
# longest polypeptide chain rather than assuming a fixed letter like 'E'.
trypsin_chain = max(
    chain_list,
    key=lambda c: len([r for r in c.get_residues() if r.get_id()[0] == ' '])
)
trypsin_chain_id = trypsin_chain.id   # Save for use in later cells
print(f"\nUsing chain '{trypsin_chain_id}' as the main trypsin polypeptide.")

# ── Amino acid composition of the trypsin chain ──────────────────────────────
print()
print(f"Amino acid composition (chain {trypsin_chain_id} = trypsin):")

# Collect all standard residues from the trypsin chain
residue_names = [res.get_resname()
                 for res in trypsin_chain.get_residues()
                 if res.get_id()[0] == ' ']

# Count each residue type using a dictionary
composition = {}
for name in residue_names:
    composition[name] = composition.get(name, 0) + 1

# Sort by count (highest first) and print top 10
sorted_composition = sorted(composition.items(), key=lambda x: x[1], reverse=True)
print(f"  Total residues: {len(residue_names)}")
print(f"  Top 10 most common:")
for residue_name, count in sorted_composition[:10]:
    percentage = (count / len(residue_names)) * 100
    print(f"    {residue_name}: {count} ({percentage:.1f}%)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Visualize the full trypsin structure in 3-D
# ─────────────────────────────────────────────────────────────────────────────

view = nv.show_structure_file(
    trypsin_path,
    representations=[{
        'type': 'cartoon',
        'params': {'colorScheme': 'sstruc'}
    }]
)

# Show calcium ion if present
view.add_ball_and_stick('_CA', color='#00CED1')

view.background = 'white'

print("Visualization key:")
print(f"  Purple/yellow ribbon = trypsin (chain {trypsin_chain_id}), colored by secondary structure")
print("  Teal sphere          = calcium ion (if present)")
print()
print("Try rotating the molecule by clicking and dragging!")
view

### Think About It — Section 1

1. **Trypsin cuts peptide bonds after Arg and Lys residues.** Look at the amino acid composition table you printed. How many Arg (ARG) and Lys (LYS) residues does trypsin itself have? Why doesn't trypsin digest itself?

2. **The active site is intact in this structure.** Look at the nglview visualization and locate where the active site might be — typically in a cleft near the center of the molecule. Based on what you know about the catalytic mechanism, what type of substrate residue do you predict fits into this pocket?

3. **Resolution matters.** This structure is solved at 1.9 Å resolution. What does that tell you about the confidence we have in the exact position of the catalytic residues?

4. **The structure is trimmed.** Real trypsin has 223 residues, but you may count fewer in the structure. Why might some residues be missing from a crystal structure?


---
## Section 2 — Finding the Catalytic Triad

### The Charge-Relay Mechanism

The three residues of the catalytic triad are not just near each other — they are connected by a **network of hydrogen bonds** that dramatically amplifies the reactivity of Ser 195:

```
Asp 102 ──H-bond──► His 57 ──H-bond──► Ser 195 ──► attacks substrate
(negative)          (base)              (nucleophile)
```

Step-by-step:
1. **His 57** accepts a proton from the **Ser 195** hydroxyl (-OH), making Ser 195's oxygen a much stronger nucleophile (O⁻)
2. **Asp 102** stabilizes the positive charge that builds up on His 57 during this transfer
3. The activated Ser 195 attacks the carbonyl carbon of the peptide bond
4. A tetrahedral intermediate forms, then collapses to release the first product (the N-terminal fragment)
5. Water enters and reverses the process to release the C-terminal fragment, regenerating the free enzyme

### Standard Residue Numbering

All serine proteases use **chymotrypsin numbering** by convention:

| Residue | Role | Atom(s) Involved |
|---------|------|-----------------|
| Ser **195** | Nucleophile | **OG** (Oγ — the serine hydroxyl oxygen) |
| His **57** | Proton shuttle | **NE2** (Nε2 — the imidazole nitrogen facing Ser 195) |
| Asp **102** | Stabilizer | **OD1** / **OD2** (carboxylate oxygens) |

The key functional distance is **Ser 195 OG → His 57 NE2**, which should be about **3.0–3.5 Å** — the ideal length for a strong hydrogen bond.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Define a function to find the catalytic triad residues
# ─────────────────────────────────────────────────────────────────────────────

def find_catalytic_triad(model, his_num=57, asp_num=102, ser_num=195):
    '''Search all chains in a model for the catalytic triad residues.

    Serine proteases use "chymotrypsin numbering" by convention.
    This function searches every chain for residues with the given
    sequence numbers and matching residue types.

    Parameters
    ----------
    model   : Bio.PDB.Model  — the structure model to search
    his_num : int            — sequence number of the histidine  (default: 57)
    asp_num : int            — sequence number of the aspartate  (default: 102)
    ser_num : int            — sequence number of the serine     (default: 195)

    Returns
    -------
    dict with keys 'His57', 'Asp102', 'Ser195'
    Each value is a Bio.PDB.Residue object, or None if not found.
    '''
    # Start with all entries as None (not found)
    triad = {'His57': None, 'Asp102': None, 'Ser195': None}

    # Loop through every chain in the model
    for chain in model.get_chains():

        # Loop through every residue in this chain
        for residue in chain.get_residues():

            # Skip non-standard residues (water, ligands, etc.)
            if residue.get_id()[0] != ' ':
                continue

            # Get the residue sequence number and three-letter name
            res_num  = residue.get_id()[1]
            res_name = residue.get_resname()

            # Check if this is the histidine of the triad
            if res_num == his_num and res_name == 'HIS':
                triad['His57'] = residue

            # Check if this is the aspartate of the triad
            elif res_num == asp_num and res_name == 'ASP':
                triad['Asp102'] = residue

            # Check if this is the serine of the triad
            elif res_num == ser_num and res_name == 'SER':
                triad['Ser195'] = residue

    return triad


print("Function find_catalytic_triad() defined.")
print("It will search all chains and return the three active-site residues.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Find the catalytic triad in trypsin and measure key distances
# ─────────────────────────────────────────────────────────────────────────────

# Find the triad using our function
triad = find_catalytic_triad(trypsin_model)

print("Catalytic Triad Search Results:")
print("-" * 45)

for label, residue in triad.items():
    if residue is not None:
        chain_id = residue.get_parent().id   # Get the chain this residue belongs to
        res_num  = residue.get_id()[1]
        res_name = residue.get_resname()
        print(f"  {label}: {res_name} {res_num} (chain {chain_id}) ✓")
    else:
        print(f"  {label}: NOT FOUND ✗")

print()

# ── Measure Cα–Cα distances (backbone geometry) ───────────────────────────────
print("Backbone (Cα–Cα) distances between triad residues:")
print("-" * 45)

# Helper to safely get the alpha-carbon of a residue
def get_ca(residue):
    '''Return the Cα atom of a residue, or None if missing.'''
    try:
        return residue['CA']
    except KeyError:
        return None

if all(r is not None for r in triad.values()):

    ca_his = get_ca(triad['His57'])
    ca_asp = get_ca(triad['Asp102'])
    ca_ser = get_ca(triad['Ser195'])

    if ca_his and ca_asp:
        dist_his_asp = calculate_distance(ca_his, ca_asp)
        print(f"  His57 Cα  ↔  Asp102 Cα : {dist_his_asp} Å")

    if ca_his and ca_ser:
        dist_his_ser = calculate_distance(ca_his, ca_ser)
        print(f"  His57 Cα  ↔  Ser195 Cα : {dist_his_ser} Å")

    if ca_asp and ca_ser:
        dist_asp_ser = calculate_distance(ca_asp, ca_ser)
        print(f"  Asp102 Cα ↔  Ser195 Cα : {dist_asp_ser} Å")

# ── Measure the KEY functional distance: Ser195 OG → His57 NE2 ───────────────
print()
print("Key functional distance (hydrogen bond distance):")
print("-" * 45)

try:
    ser_og  = triad['Ser195']['OG']    # Serine hydroxyl oxygen (the nucleophile)
    his_ne2 = triad['His57']['NE2']    # Histidine imidazole nitrogen facing Ser
    functional_dist = calculate_distance(ser_og, his_ne2)
    print(f"  Ser195 OG ↔ His57 NE2 : {functional_dist} Å")
    print()
    if functional_dist <= 3.5:
        print("  → Within hydrogen-bond distance (≤ 3.5 Å) ✓")
        print("  → This confirms the charge-relay network is intact.")
    else:
        print("  → Distance > 3.5 Å — the inhibitor may have perturbed the geometry.")
except KeyError as e:
    print(f"  Could not find atom {e} — check the PDB file for alternate conformations.")

# ── Also measure Asp102 OD1 → His57 ND1 (the other side of the relay) ────────
try:
    asp_od1 = triad['Asp102']['OD1']
    his_nd1 = triad['His57']['ND1']
    asp_his_dist = calculate_distance(asp_od1, his_nd1)
    print(f"  Asp102 OD1 ↔ His57 ND1: {asp_his_dist} Å")
except KeyError:
    pass

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Visualize the catalytic triad in 3-D
# ─────────────────────────────────────────────────────────────────────────────

def make_ngl_selection(residue):
    '''Return an NGL selection string for a single residue, e.g. "195:A".'''
    res_num  = residue.get_id()[1]
    chain_id = residue.get_parent().id
    return f"{res_num}:{chain_id}"

# Gray cartoon backdrop at low opacity — same pattern as notebook 03
view2 = nv.show_structure_file(
    trypsin_path,
    representations=[{
        'type': 'cartoon',
        'params': {'colorScheme': 'sstruc', 'opacity': 0.4}
    }]
)

# Show each triad residue as colored ball-and-stick
# Color scheme: Ser=green (nucleophile), His=blue (shuttle), Asp=red (stabilizer)
if triad['Ser195'] is not None:
    view2.add_ball_and_stick(make_ngl_selection(triad['Ser195']), color='#1A9641')

if triad['His57'] is not None:
    view2.add_ball_and_stick(make_ngl_selection(triad['His57']), color='#2166AC')

if triad['Asp102'] is not None:
    view2.add_ball_and_stick(make_ngl_selection(triad['Asp102']), color='#D7191C')

view2.background = 'white'

print("Active site visualization:")
print("  Green sticks = Ser 195 (nucleophile)")
print("  Blue sticks  = His 57 (proton shuttle)")
print("  Red sticks   = Asp 102 (stabilizer)")
print("  Faded ribbon = rest of trypsin (context)")
print()
print("Rotate until you can see all three triad residues.")
print("Notice how they form a nearly straight line: Asp102 → His57 → Ser195")
view2

### Think About It — Section 2

1. **Ser 195 OG → His 57 NE2 distance.** What distance did you measure? Is it within hydrogen-bond range (≤ 3.5 Å)? What does this tell you about the readiness of the enzyme for catalysis?

2. **The "charge relay."** Asp 102 is negatively charged at physiological pH. His 57 is normally uncharged. Ser 195 is normally a poor nucleophile. Explain in plain language how connecting these three residues transforms Ser 195 into a powerful nucleophile.

3. **Mutation experiment.** If you mutated Ser 195 → Ala (removing the hydroxyl group), what would happen to enzyme activity? What if you mutated His 57 → Ala? Would the effects be the same or different?

4. **Inhibitor location.** In the visualization, where does the BPTI inhibitor sit relative to the triad? What does its position tell you about how inhibitors can be designed to block protease activity?

---
## Section 3 — Sequence Conservation: Trypsin vs. Chymotrypsin

### Why Compare Sequences?

Trypsin and chymotrypsin perform the same chemical reaction (cleave peptide bonds) using the same catalytic mechanism (the Ser–His–Asp triad), but they cut at **different places** in a protein:

| Enzyme | Cuts after… | Why? |
|--------|-------------|------|
| Trypsin | Arg, Lys (positive) | Its "specificity pocket" is negatively charged |
| Chymotrypsin | Phe, Trp, Tyr (large, hydrophobic) | Its pocket is large and hydrophobic |

Despite this difference, they share ~**40% sequence identity** — meaning 40% of their amino acids are the same at the same positions. This is strong evidence they **evolved from a common ancestor** (divergent evolution).

### What We Expect to Find

- **High conservation** around the catalytic triad residues (His 57, Asp 102, Ser 195)
- **Low conservation** around the "specificity pocket" (where substrate recognition occurs)
- A global alignment score that reflects their shared ancestry

### The Alignment Algorithm

We will use the **Needleman–Wunsch global alignment** algorithm (`pairwise2.align.globalms`).
It finds the optimal alignment by maximizing a score based on:
- **+2** for each matched position
- **−1** for each mismatch
- **−10** penalty to open a gap (insertion or deletion)
- **−0.5** penalty for each additional gap position

These penalties strongly discourage gaps, which is appropriate when we know the sequences have similar lengths.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — Load chymotrypsin (4CHA) and align with trypsin
# ─────────────────────────────────────────────────────────────────────────────

def extract_sequence(model):
    '''Extract the full amino acid sequence from a structure model.

    Uses BioPython's PPBuilder which intelligently connects polypeptide
    fragments by checking for peptide-bond geometry.

    Parameters
    ----------
    model : Bio.PDB.Model

    Returns
    -------
    str : single-letter amino acid sequence (all chains concatenated)
    '''
    builder = PPBuilder()
    sequence_parts = []

    # get_peptides() returns a list of Polypeptide objects
    # (each is a continuous stretch without chain breaks)
    for polypeptide in builder.build_peptides(model):
        sequence_parts.append(str(polypeptide.get_sequence()))

    full_sequence = ''.join(sequence_parts)
    return full_sequence


# ── Load chymotrypsin ─────────────────────────────────────────────────────────
print("Loading chymotrypsin (4CHA)...")
chymo_structure, chymo_model, chymo_path = load_structure('4CHA')

# ── Extract sequences ─────────────────────────────────────────────────────────
print()
print("Extracting sequences...")
trypsin_seq = extract_sequence(trypsin_model)
chymo_seq   = extract_sequence(chymo_model)

print(f"Trypsin  (1TGN) sequence length: {len(trypsin_seq)} residues")
print(f"Chymo    (4CHA) sequence length: {len(chymo_seq)} residues")

# ── Run global pairwise alignment ─────────────────────────────────────────────
print()
print("Running Needleman–Wunsch global alignment...")

# globalms(seq1, seq2, match_score, mismatch_score, gap_open, gap_extend)
alignments = pairwise2.align.globalms(
    trypsin_seq,   # sequence 1
    chymo_seq,     # sequence 2
    2,             # reward for a match
    -1,            # penalty for a mismatch
    -10,           # penalty to open a gap
    -0.5           # penalty for each additional gap position
)

# Take the best (highest-scoring) alignment
best_alignment = alignments[0]
aln_trypsin = best_alignment.seqA   # Aligned trypsin sequence (with gaps as '-')
aln_chymo   = best_alignment.seqB   # Aligned chymotrypsin sequence (with gaps as '-')
aln_score   = best_alignment.score

# ── Calculate identity ────────────────────────────────────────────────────────
matches   = sum(1 for a, b in zip(aln_trypsin, aln_chymo) if a == b and a != '-')
aln_len   = len(aln_trypsin)
identity  = (matches / aln_len) * 100

print(f"Alignment length: {aln_len} positions")
print(f"Identical positions: {matches}")
print(f"Sequence identity: {identity:.1f}%")
print(f"Alignment score:  {aln_score:.1f}")
print()
print(f"These enzymes are {identity:.1f}% identical — strong evidence for")
print("a common ancestral serine protease (divergent evolution).")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 — Print a readable alignment showing conservation
# ─────────────────────────────────────────────────────────────────────────────

def print_alignment_block(seq_a, seq_b, label_a='Trypsin ', label_b='ChymoTr ',
                           block_width=60):
    '''Print a pairwise alignment in block format with a conservation line.

    Parameters
    ----------
    seq_a, seq_b : str  — aligned sequences (with '-' for gaps)
    label_a, label_b : str — labels for each sequence (pad to same length)
    block_width : int   — number of alignment columns per printed block
    '''
    print(f"Alignment: {label_a.strip()} vs {label_b.strip()}")
    print(f"Legend: | = identical,  . = different,  - = gap")
    print()

    for start in range(0, len(seq_a), block_width):
        end = start + block_width

        # Slice this block
        block_a = seq_a[start:end]
        block_b = seq_b[start:end]

        # Build conservation line: '|' for identical, '.' for different
        conservation = ''
        for a, b in zip(block_a, block_b):
            if a == b and a != '-':
                conservation += '|'
            else:
                conservation += '.'

        # Print the block
        print(f"{label_a}{block_a}")
        print(f"        {conservation}")
        print(f"{label_b}{block_b}")
        print()


# Print the alignment (it may be long — scroll to see all of it)
print_alignment_block(aln_trypsin, aln_chymo)

### Think About It — Section 3

1. **Sequence identity.** What percent sequence identity did you find between trypsin and chymotrypsin? Is this more or less than you expected for two enzymes that use the same mechanism?

2. **Catalytic triad conservation.** Are His 57, Asp 102, and Ser 195 conserved between the two sequences? What would happen to enzyme function if any of these were mutated?

3. **Divergent evolution.** The fact that these enzymes share ~40% identity and the same mechanism suggests they evolved from a common ancestral protease. What selective pressure might have driven the evolution of different substrate specificities (Arg/Lys vs. Phe/Trp/Tyr)?

4. **Gap penalties.** The alignment uses a gap-open penalty of −10 and gap-extension of −0.5. Why do we penalize gaps? What biological event do gaps in a protein alignment correspond to?

---
## Section 4 — Master Analysis Pipeline

### Building Reusable Code

So far we have written individual functions for loading structures, finding the triad, and comparing sequences.
Now we combine them into a **single master function** that can analyze any serine protease in one call.

This is the heart of **modular programming**: once you have tested each component separately, you can compose them into more powerful tools.

### What the Pipeline Does

```
analyze_serine_protease(pdb_id)
    │
    ├─► load_structure()         → downloads and parses the PDB file
    ├─► basic stats              → counts chains, residues, atoms
    ├─► amino acid composition   → classifies residues by chemistry
    ├─► find_catalytic_triad()   → locates His, Asp, Ser
    ├─► distance measurements    → Cα–Cα and functional distances
    └─► extract_sequence()       → returns the sequence for later comparison
```

The function returns a **dictionary** of results — a key–value store — so you can easily access any result by name (e.g., `result['identity']`, `result['ser_his_dist']`).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 19 — Define the master analysis function
# ─────────────────────────────────────────────────────────────────────────────

def analyze_serine_protease(pdb_id, his_num=57, asp_num=102, ser_num=195,
                             file_dir='.'):
    '''Complete analysis pipeline for a serine protease structure.

    Loads the structure, computes basic statistics, locates the catalytic
    triad, measures key distances, and extracts the sequence.

    Parameters
    ----------
    pdb_id   : str  — 4-character PDB accession code
    his_num  : int  — residue number of the histidine  (default: 57)
    asp_num  : int  — residue number of the aspartate  (default: 102)
    ser_num  : int  — residue number of the serine     (default: 195)
    file_dir : str  — directory for downloaded PDB files

    Returns
    -------
    results  : dict  — all computed metrics
    model    : Bio.PDB.Model
    pdb_path : str
    sequence : str
    '''
    print(f"\n{'='*55}")
    print(f"Analyzing {pdb_id}...")
    print(f"{'='*55}")

    # ── Step 1: Load structure ────────────────────────────────────────────────
    structure, model, pdb_path = load_structure(pdb_id, file_dir=file_dir)

    # ── Step 2: Count chains and residues ─────────────────────────────────────
    chain_list = list(model.get_chains())

    total_residues = 0
    for chain in chain_list:
        aa = [r for r in chain.get_residues() if r.get_id()[0] == ' ']
        total_residues += len(aa)

    total_atoms = len(list(model.get_atoms()))

    print(f"  Chains: {len(chain_list)} | Residues: {total_residues} | Atoms: {total_atoms}")

    # ── Step 3: Amino acid composition ───────────────────────────────────────
    all_res_names = []
    for chain in chain_list:
        for residue in chain.get_residues():
            if residue.get_id()[0] == ' ':
                all_res_names.append(residue.get_resname())

    class_counts = {'Nonpolar': 0, 'Polar': 0, 'Positive': 0, 'Negative': 0, 'Other': 0}
    for name in all_res_names:
        cls = classify_residue(name)
        class_counts[cls] = class_counts.get(cls, 0) + 1

    # ── Step 4: Find the catalytic triad ──────────────────────────────────────
    triad = find_catalytic_triad(model, his_num=his_num,
                                  asp_num=asp_num, ser_num=ser_num)

    triad_found = sum(1 for r in triad.values() if r is not None)
    print(f"  Catalytic triad: {triad_found}/3 residues found")

    # ── Step 5: Measure distances ─────────────────────────────────────────────
    ser_his_dist = None
    his_asp_dist = None

    if triad['Ser195'] and triad['His57']:
        try:
            ser_his_dist = calculate_distance(triad['Ser195']['OG'],
                                               triad['His57']['NE2'])
            print(f"  Ser{ser_num} OG ↔ His{his_num} NE2 : {ser_his_dist} Å")
        except KeyError:
            # Some structures may lack the OG or NE2 atom (e.g. modified residue)
            pass

    if triad['His57'] and triad['Asp102']:
        try:
            his_asp_dist = calculate_distance(triad['His57']['CA'],
                                               triad['Asp102']['CA'])
        except KeyError:
            pass

    # ── Step 6: Extract sequence ──────────────────────────────────────────────
    sequence = extract_sequence(model)
    print(f"  Sequence length: {len(sequence)} residues")

    # ── Assemble results dictionary ───────────────────────────────────────────
    results = {
        'pdb_id':         pdb_id,
        'n_chains':       len(chain_list),
        'n_residues':     total_residues,
        'n_atoms':        total_atoms,
        'composition':    class_counts,
        'triad_found':    triad_found,
        'triad':          triad,
        'ser_his_dist':   ser_his_dist,
        'his_asp_dist':   his_asp_dist,
        'seq_length':     len(sequence),
    }

    return results, model, pdb_path, sequence


print("Master function analyze_serine_protease() defined.")
print("Ready to run on any serine protease PDB structure.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 20 — Run the master pipeline on trypsin and chymotrypsin; compare
# ─────────────────────────────────────────────────────────────────────────────

# Run the pipeline on both enzymes
# Note: we already loaded these structures earlier, but the function
# will just re-use the already-downloaded files
trypsin_results, _, _, trypsin_seq2 = analyze_serine_protease('1TGN')
chymo_results,   _, _, chymo_seq2   = analyze_serine_protease('4CHA')

# ── Build comparison table ────────────────────────────────────────────────────
print()
print("=" * 55)
print("COMPARISON: TRYPSIN vs. CHYMOTRYPSIN")
print("=" * 55)

comparison_data = {
    'Metric': [
        'PDB ID',
        'Total residues',
        'Total atoms',
        'Triad residues found',
        'Ser–His H-bond (Å)',
        'Sequence length',
    ],
    'Trypsin (1TGN)': [
        trypsin_results['pdb_id'],
        trypsin_results['n_residues'],
        trypsin_results['n_atoms'],
        f"{trypsin_results['triad_found']}/3",
        trypsin_results['ser_his_dist'],
        trypsin_results['seq_length'],
    ],
    'Chymotrypsin (4CHA)': [
        chymo_results['pdb_id'],
        chymo_results['n_residues'],
        chymo_results['n_atoms'],
        f"{chymo_results['triad_found']}/3",
        chymo_results['ser_his_dist'],
        chymo_results['seq_length'],
    ],
}

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

# ── Bar chart: amino acid class composition side by side ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Amino Acid Composition: Trypsin vs. Chymotrypsin', fontsize=13)

class_order  = ['Nonpolar', 'Polar', 'Positive', 'Negative']
class_colors = ['#4393C3', '#74C476', '#D7191C', '#F4A811']

for ax, results, title in zip(
        axes,
        [trypsin_results, chymo_results],
        ['Trypsin (1TGN)', 'Chymotrypsin (4CHA)']):

    comp  = results['composition']
    total = sum(comp[k] for k in class_order)
    pcts  = [(comp[k] / total) * 100 for k in class_order]

    bars = ax.bar(class_order, pcts, color=class_colors, edgecolor='white',
                  linewidth=0.8)

    # Add percentage labels on top of each bar
    for bar, pct in zip(bars, pcts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Percentage of residues (%)')
    ax.set_ylim(0, max(pcts) * 1.2)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('composition_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as composition_comparison.png")

### Think About It — Section 4

1. **Composition differences.** Look at the amino acid composition bar charts. Trypsin cleaves after *positively charged* residues (Arg, Lys). Does trypsin have a higher proportion of negative residues in its binding pocket than chymotrypsin? (Hint: overall composition reflects the whole protein, not just the active site — but the trend should still be visible.)

2. **Ser–His distance comparison.** How do the Ser 195 OG → His 57 NE2 distances compare between trypsin and chymotrypsin? Are they within the expected range for a hydrogen bond?

3. **Modular design advantage.** This notebook now has a single function `analyze_serine_protease()` that can process any serine protease. What are the advantages of this design over writing the same code twice? Can you think of a case where this modular approach could fail?

4. **Applying the pipeline.** In the next section you will apply this pipeline to an unknown enzyme. Before running it: what results would you expect if the enzyme is a serine protease? What if it's *not* a serine protease?

---
## Capstone Exercise — The Mystery Enzyme

### Your Challenge

You have been given a PDB accession code for an enzyme of unknown identity: **`1SBT`**

Your task is to use all the tools built in this notebook to answer the following questions:

1. **What protein is 1SBT?** (Load it and check the printed output)
2. **Does it have a catalytic triad?** Run `find_catalytic_triad()` with the *default* His 57 / Asp 102 / Ser 195 numbers. Does it find all three residues?
3. **If not, search for the triad at different positions.** Try `his_num=64, asp_num=32, ser_num=221` instead.
4. **Is it a serine protease?** Based on the triad, the Ser–His distance, and any other evidence you find.
5. **Is it homologous to trypsin?** Align its sequence to trypsin and compute percent identity.
6. **What does the comparison tell you about evolution?**

---

### The Twist: Convergent Evolution

> *Hint: 1SBT may not be what you expect.*

Some enzymes independently evolved the same catalytic solution — the serine protease triad — even though they share **no sequence homology** and have **completely different protein folds**.

This is called **convergent evolution**: different lineages arriving at the same functional solution independently, just as dolphins and sharks independently evolved streamlined body shapes.

| Feature | Trypsin / Chymotrypsin | Mystery Enzyme |
|---------|----------------------|----------------|
| Triad chemistry | Ser–His–Asp | ? |
| Triad positions | Ser 195, His 57, Asp 102 | ? |
| Sequence identity to trypsin | 100% / ~40% | ? |
| Evolutionary origin | Common ancestral protease | ? |

### Instructions

Use the 4 blank code cells below to:
1. Load 1SBT and print basic stats
2. Search for the catalytic triad (try both default and alternate positions)
3. Visualize the active site in nglview
4. Align to trypsin and compute percent identity

**Guiding questions to answer in a markdown cell (add one if you like):**
- What organism does 1SBT come from?
- Where are the catalytic residues located (chain and number)?
- What is the percent identity to trypsin?
- Does the similar triad geometry (Ser–His distance) suggest shared ancestry or convergent evolution?
- What is the structural/biological significance of finding the same triad in two completely unrelated proteins?

In [ ]:
# Student code cell 1 of 4
# Add your code here


In [ ]:
# Student code cell 2 of 4
# Add your code here


In [ ]:
# Student code cell 3 of 4
# Add your code here


In [ ]:
# Student code cell 4 of 4
# Add your code here


---
## Final Summary — Five Levels of Protein Structure

Congratulations on completing all five notebooks! Here is what you have learned:

| Notebook | Level | Key Concept | Tool Used |
|----------|-------|------------|-----------|
| 1 | Primary | Sequence, composition, motifs | BioPython SeqIO, pandas |
| 2 | Secondary | Helix/sheet assignment, visualization | DSSP, nglview |
| 3 | Tertiary | Domains, disulfide bonds, atomic distances | PDB parser, matplotlib |
| 4 | Quaternary | Subunit interfaces, cooperativity | Interface analysis, Hill equation |
| 5 | Integration | Enzyme active sites, conservation, evolution | Alignment, master pipeline |

### Key Biological Themes

**Structure determines function.** The precise three-dimensional arrangement of Ser 195, His 57, and Asp 102 — positioned within 3–4 Å of each other — creates the charge-relay mechanism that makes serine proteases among the most efficient catalysts in biology.

**Conservation reveals importance.** When a residue is conserved across millions of years of evolution, it is almost always because mutations at that position are lethal. The near-perfect conservation of the catalytic triad across all serine proteases is direct evidence of this principle.

**Evolution works with what it has.** Both divergent evolution (trypsin and chymotrypsin from a common ancestor) and convergent evolution (subtilisin independently inventing the same triad) are visible in protein structures. Sequence alignment lets us distinguish between the two.

### Further Reading

| Resource | What It's For |
|----------|---------------|
| [RCSB PDB](https://www.rcsb.org) | Browse and download any protein structure |
| [UniProt](https://www.uniprot.org) | Protein sequences and functional annotation |
| [Pfam](https://www.ebi.ac.uk/interpro) | Protein domain families and alignments |
| [BLAST](https://blast.ncbi.nlm.nih.gov) | Find proteins with similar sequences |
| [PyMOL](https://pymol.org) | Advanced molecular visualization |
| BioPython Tutorial | https://biopython.org/DIST/docs/tutorial/Tutorial.html |

### Looking Ahead

The methods you have used in these notebooks — structure loading, distance measurement, sequence alignment, visualization — are the same methods used in:
- Drug discovery (finding enzyme inhibitor binding sites)
- Protein engineering (designing new enzymes)
- Evolutionary biology (reconstructing ancestral proteins)
- Clinical diagnostics (understanding disease-causing mutations)

You now have the computational foundation to explore any of these directions.

---
*Bio 4525 — Structural Bioinformatics of Proteins | Washington University in St. Louis*